In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Loan_Default_Prediction\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Loan_Default_Prediction'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainingConfig:
    root_dir: Path
    train_file_path: Path
    validation_file_path: Path
    test_file_path: Path
    model_file_path: Path
    metrics_file_path: Path
    model_params: dict
    target_column: str

In [6]:
from src.loan_default_prediction.constants import *
from src.loan_default_prediction.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> ModelTrainingConfig:
        config = self.config.model_training

        create_directories([config.root_dir])

        model_training_config = ModelTrainingConfig(
            root_dir=config.root_dir,
            train_file_path=config.train_file_path,
            validation_file_path=config.validation_file_path,
            test_file_path=config.test_file_path,
            model_file_path=config.model_file_path,
            metrics_file_path=config.metrics_file_path,
            model_params=dict(self.params.model_params),
            target_column=config.target_column
        )

        return model_training_config    

In [8]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from loan_default_prediction import logger
from src.loan_default_prediction.entity.config_entity import ModelTrainingConfig
from src.loan_default_prediction.utils.common import save_bin, save_json

In [9]:
from pathlib import Path

class ModelTraining:
    def __init__(self, config: ModelTrainingConfig):
        self.config = config

    def load_split_data(self, file_path: str) -> pd.DataFrame:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Split data file not found: {file_path}")

        logger.info(f"Loading split data from: {file_path}")
        return pd.read_csv(file_path)

    def train_model(self, train_df: pd.DataFrame, val_df: pd.DataFrame) -> XGBClassifier:
        target_col = self.config.target_column
        if target_col not in train_df.columns or target_col not in val_df.columns:
            raise KeyError(f"Target column '{target_col}' not found in train/validation data")

        X_train = train_df.drop(columns=[target_col])
        y_train = train_df[target_col]
        X_val = val_df.drop(columns=[target_col])
        y_val = val_df[target_col]

        logger.info("Initializing XGBoost classifier")
        model = XGBClassifier(**self.config.model_params)

        logger.info("Starting XGBoost training with early stopping")
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        logger.info("Model training completed")
        return model

    def evaluate_model(self, model: XGBClassifier, df: pd.DataFrame) -> dict:
        target_col = self.config.target_column
        X = df.drop(columns=[target_col])
        y = df[target_col]

        y_pred = model.predict(X)
        y_proba = model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else None

        metrics = {
            "accuracy": accuracy_score(y, y_pred),
            "precision": precision_score(y, y_pred, zero_division=0),
            "recall": recall_score(y, y_pred, zero_division=0),
            "f1_score": f1_score(y, y_pred, zero_division=0),
        }

        if y_proba is not None:
            metrics["roc_auc"] = roc_auc_score(y, y_proba)

        return {k: float(v) for k, v in metrics.items()}

    def save_model(self, model: XGBClassifier) -> None:
        model_path = Path(self.config.model_file_path)
        model_path.parent.mkdir(parents=True, exist_ok=True)
        save_bin(data=model, path=model_path)
        logger.info(f"Saved trained model at: {model_path}")

    def save_metrics(self, metrics: dict) -> None:
        metrics_path = Path(self.config.metrics_file_path)
        metrics_path.parent.mkdir(parents=True, exist_ok=True)
        save_json(path=metrics_path, data=metrics)
        logger.info(f"Saved training metrics at: {metrics_path}")

    def initiate_model_training(self) -> bool:
        try:
            train_df = self.load_split_data(self.config.train_file_path)
            val_df = self.load_split_data(self.config.validation_file_path)
            # test_df = self.load_split_data(self.config.test_file_path)

            model = self.train_model(train_df, val_df)
            self.save_model(model)

            logger.info("Evaluating model on validation data")
            validation_metrics = self.evaluate_model(model, val_df)
            # test_metrics = self.evaluate_model(model, test_df)

            metrics = {
                "validation": validation_metrics,
                # "test": test_metrics,
                "best_iteration": int(model.get_booster().best_iteration) if hasattr(model, "get_booster") and model.get_booster().best_iteration != -1 else None
            }

            self.save_metrics(metrics)
            logger.info("Model training stage completed successfully")
            return True
        except Exception as e:
            logger.error(f"Error during model training: {str(e)}")
            raise e


In [10]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_training_config()
    model_trainer_config = ModelTraining(config=model_trainer_config)
    model_trainer_config.initiate_model_training()
except Exception as e:
    raise e

[2026-05-03 04:15:00,884: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-03 04:15:00,888: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-03 04:15:00,889: INFO: common: created directory at: artifacts]
[2026-05-03 04:15:00,891: INFO: common: created directory at: artifacts/model_training]
[2026-05-03 04:15:00,891: INFO: 977156298: Loading split data from: artifacts/data_transformation/split_data/train.csv]


[2026-05-03 04:15:03,814: INFO: 977156298: Loading split data from: artifacts/data_transformation/split_data/validation.csv]
[2026-05-03 04:15:04,122: INFO: 977156298: Initializing XGBoost classifier]
[2026-05-03 04:15:04,124: INFO: 977156298: Starting XGBoost training with early stopping]


c:\Users\Hp\anaconda3\envs\loan\Lib\site-packages\xgboost\callback.py:385: UserWarning: [04:15:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[2026-05-03 04:15:06,715: INFO: 977156298: Model training completed]
[2026-05-03 04:15:06,749: INFO: common: binary file saved at: artifacts\model_training\xgboost_model.pkl]
[2026-05-03 04:15:06,751: INFO: 977156298: Saved trained model at: artifacts\model_training\xgboost_model.pkl]
[2026-05-03 04:15:06,755: INFO: 977156298: Evaluating model on validation data]
[2026-05-03 04:15:06,882: INFO: common: json file saved at: artifacts\model_training\model_metrics.json]
[2026-05-03 04:15:06,882: INFO: 977156298: Saved training metrics at: artifacts\model_training\model_metrics.json]
[2026-05-03 04:15:06,885: INFO: 977156298: Model training stage completed successfully]
